# CSS repetition-code decoding

This notebook decodes one CSS sector of a small GKP repetition code. It mirrors the surface-code CSS example: build one sector, run belief propagation on one displacement, check the residual, then run a compact distance sweep.

In [ ]:
using Random
using LinearAlgebra
using LatticeDecoder

Random.seed!(4);

Build a distance-3 repetition code and select one CSS sector. With `bit_flip = false`, the protected qubit is encoded in the momentum basis, so this example decodes the reduced momentum-sector check matrix.

In [ ]:
d = 3
bit_flip = false

code = GKP_Rep_Code(d, bit_flip, true);
M = code.code;

if bit_flip
    H = M[1:d, 1:d]
else
    H = M[(d + 1):end, (d + 1):end]
end

G = round.(sqrt(2) * inv(H)) / sqrt(2);
logical_check = inv(H);

(check_matrix_size = size(H), generator_size = size(G))

Draw one Gaussian displacement, run serial belief propagation, then convert the continuous BP estimate into an integer correction.

In [ ]:
noise_std = 0.05
max_iter = size(H, 2)
decoder = "lsd"
search_interval = 1.0

error_vector = sample_error(noise_std, size(H, 2));
received = copy(error_vector);

tanner_graph = initialize_tanner_graph(H);
bp_estimate = run_serial_belief_propagation!(
    tanner_graph,
    received,
    noise_std,
    max_iter,
    decoder;
    search_interval = search_interval,
);

decoded_integer_correction = hard_decision(bp_estimate, H);

Check whether the residual displacement is trivial up to the decoded sector lattice.

In [ ]:
function is_not_logical_error(logical_check, residual; atol = 1e-5)
    logical_coordinates = logical_check' * residual
    return all(abs(x - round(x)) < atol for x in logical_coordinates)
end

correction = received - G * decoded_integer_correction;
residual = error_vector - correction;
logical_success = is_not_logical_error(logical_check, residual)

## Small distance sweep

The next cells run a small CSS-sector sweep for `d = 3, 5, 7` over five noise values and plot the observed logical failure rate. This is intentionally small so the notebook remains an example; increase `samples_per_point` for smoother curves.

In [ ]:
using Plots

function css_rep_code_sector(d; bit_flip = false)
    code = GKP_Rep_Code(d, bit_flip, true)
    M = code.code

    if bit_flip
        H = M[1:d, 1:d]
    else
        H = M[(d + 1):end, (d + 1):end]
    end

    G = round.(sqrt(2) * inv(H)) / sqrt(2)
    return H, G
end

function css_decode_fails(H, G, noise_std; decoder = "nearest", search_interval = 1.0)
    logical_check = inv(H)
    error_vector = sample_error(noise_std, size(H, 2))
    received = copy(error_vector)
    tanner_graph = initialize_tanner_graph(H)

    bp_estimate = run_serial_belief_propagation!(
        tanner_graph,
        received,
        noise_std,
        size(H, 2),
        decoder;
        search_interval = search_interval,
    )

    decoded_integer_correction = hard_decision(bp_estimate, H)
    correction = received - G * decoded_integer_correction
    residual = error_vector - correction

    return !is_not_logical_error(logical_check, residual)
end

In [ ]:
distances = [3, 5, 7]
sigmas = collect(range(0.3, 0.7; length = 5)) ./ sqrt(2 * pi)
samples_per_point = 50_000

failure_rates = Dict{Int, Vector{Float64}}()

for distance in distances
    H_d, G_d = css_rep_code_sector(distance; bit_flip = bit_flip)
    rates = Float64[]

    for sigma in sigmas
        failures = count(_ -> css_decode_fails(H_d, G_d, sigma), 1:samples_per_point)
        push!(rates, failures / samples_per_point)
    end

    failure_rates[distance] = rates
end

for d in distances
    println("Distance $d failure rates: ", failure_rates[d])
end

In [ ]:
p = plot(
    xlabel = "noise std σ",
    ylabel = "logical failure rate",
    title = "CSS repetition-code decoding",
    legend = :topleft,
    yscale = :log10,
)

for distance in distances
    plot!(p, sigmas * sqrt(2 * pi), failure_rates[distance]; marker = :circle, label = "d = $(distance)")
end
p